> **Pravalika - TRANSFORM B: FEATURE ENGINEERING & LOAD**: this notebook has only my section. In the final `01_ETL_Pipeline.ipynb`, it is pasted in order after the earlier sections, and everything runs top to bottom.

# PerfectPetClub – Customer Churn ETL Pipeline

**Module:** Data Engineering Team Presentation &nbsp;|&nbsp; **Notebook 1 of 2:** ETL pipeline

**Goal:** build a clean, one-row-per-customer dataset for analysing **customer churn**, where churn means a customer has **cancelled the auto-renew** of their subscription (`subscription.auto_renew == 0`).

| Section | Owner | What it does |
|---|---|---|
| 0. Setup & configuration | Team | Imports, paths, logging |
| 1. Extract | Bhavana Padala | Read SQLite DB, CRM CSV, server log CSV, survey CSV + question metadata |
| 2. Transform A – data quality & cleaning | Siddu Gundeboina | Data quality checks, type conversion, parsing, cleaning |
| 3. Transform B – feature engineering | Pravalika | Build customer-level features and the churn target |
| 4. Load | Pravalika | Write and verify the Parquet files |
| 5. Testing & validation | Denya Sree Adapa | Unit tests and end-to-end checks on the output |
| Repo, README, merge, HTML export | Ramya Sree Kasula & Denya Sree Adapa | Git repository, merging branches, run-all + HTML export |

**Architecture**

```
 SQLite DB ─┐
 CRM CSV  ──┤   extract_*()  ──►  raw dict  ──►  run_quality_checks() ──► clean_*()  ──►  build_*_features()
 Server log ┤                                   (DQ report)               (typed,         (one row per customer)
 Survey CSV ┘                                                              parsed)               │
                                                                                                  ▼
                                                               load_parquet()  ──►  output/customer_churn.parquet
                                                                                    output/customer_churn_model_ready.parquet
```

## 0. Setup & configuration

In [ ]:
import ast
import logging
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# ---- Configuration (change paths here, not inside functions) ----
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCES = {
    "db": DATA_DIR / "PerfectPetClub.db",
    "crm": DATA_DIR / "PerfectPetClub_CRM_report_1.csv",
    "server_log": DATA_DIR / "PerfectPetClub_server_log_1.csv",
    "survey": DATA_DIR / "PerfectPetClub_survey.csv",
    "survey_questions": DATA_DIR / "PerfectPetClub_survey_questions.txt",
}
DB_TABLES = ["customer", "subscription", "subscription_type", "sale", "sale_product",
             "product", "product_category", "payment", "product_review"]

OUTPUT_FILE = OUTPUT_DIR / "customer_churn.parquet"
MODEL_READY_FILE = OUTPUT_DIR / "customer_churn_model_ready.parquet"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
log = logging.getLogger("ppc_etl")
log.info("pandas %s | numpy %s", pd.__version__, np.__version__)

## 3. Transform B: feature engineering  *(owner: Pravalika)*

**Grain:** one row per customer (221 customers).  **Target:** `churned = 1` if `auto_renew == 0`.

**Snapshot date:** the latest timestamp in the data (last server-log or sale event). Every "recency" and "tenure" feature is measured up to this date.

**Key design decision (normalising by tenure):** customers who have been with us longer naturally have *more* orders, logins and
CRM contacts. Raw counts would therefore look like "loyalty" when they are really just tenure. So the main behavioural features are
**rates per active month** (`orders_per_month`, `logins_per_month`, `complaints_per_month`).

| Feature group | Source | Examples |
|---|---|---|
| Subscription | DB | `subscription_type`, `delivery_service`, `tenure_days`, `start_month` |
| Purchasing | DB sale/product/payment | `orders_per_month`, `avg_order_value_gbp`, `n_categories`, `days_since_last_order` |
| Delivery | DB sale | `late_delivery_rate`, `avg_delivery_delay_days` |
| Reviews | DB product_review | `avg_star_rating`, `low_rating_share` |
| Customer service | CRM CSV | `n_complaints`, `complaints_per_month`, `complaint_share`, `main_contact_platform` |
| Engagement | Server log | `logins_per_month`, `main_device_type`, `n_devices`, `days_since_last_login` |
| Survey | Survey CSV | `income_gbp`, `income_band`, `nps_recommend`, `nps_category`, `online_comparison_freq` |

In [ ]:
def get_snapshot_date(clean: dict) -> pd.Timestamp:
    return max(clean["server_log"]["datetime"].max(), clean["sale"]["order_date"].max())


def build_subscription_features(clean: dict, snapshot: pd.Timestamp) -> pd.DataFrame:
    df = clean["customer"].merge(clean["subscription"], on="subscription_id", how="left", validate="one_to_one")
    active_end = df["end_date"].where(df["end_date"] < snapshot, snapshot)        # churned customers stop at end_date
    df["tenure_days"] = (active_end - df["start_date"]).dt.days.clip(lower=1)
    df["tenure_months"] = df["tenure_days"] / 30.44
    df["start_month"] = df["start_date"].dt.to_period("M").astype(str)
    df["end_month"] = df["end_date"].dt.to_period("M").astype(str)
    df["churned"] = (df["auto_renew"] == 0).astype(int)
    return df[["customer_id", "email", "county", "subscription_type", "delivery_service", "start_date", "end_date",
               "start_month", "end_month", "tenure_days", "tenure_months", "auto_renew", "churned"]]


def build_sales_features(clean: dict, snapshot: pd.Timestamp) -> pd.DataFrame:
    sale, sp = clean["sale"], clean["sale_product"]
    shop = sp[~sp["is_subscription_item"]]                                        # exclude the monthly subscription charge
    shop_sales = sale[sale["sale_id"].isin(shop["sale_id"])].copy()
    shop_sales["delay_days"] = (shop_sales["delivered_date"] - shop_sales["expected_date"]).dt.total_seconds() / 86400
    shop_sales["is_late"] = shop_sales["delay_days"] > 0

    f = shop_sales.groupby("customer_id").agg(
        n_orders=("sale_id", "nunique"),
        total_spend_gbp=("amount_gbp", "sum"),
        avg_order_value_gbp=("amount_gbp", "mean"),
        last_order_date=("order_date", "max"),
        late_delivery_rate=("is_late", "mean"),
        avg_delivery_delay_days=("delay_days", "mean"),
        favourite_payment=("payment_gateway", lambda s: s.mode().iat[0]),
    )
    items = shop.merge(sale[["sale_id", "customer_id"]], on="sale_id")
    f = f.join(items.groupby("customer_id").agg(n_items=("quantity", "sum"), n_categories=("category", "nunique"),
                                                favourite_category=("category", lambda s: s.mode().iat[0])))
    f["days_since_last_order"] = (snapshot - f.pop("last_order_date")).dt.days
    return f.reset_index()


def build_review_features(clean: dict) -> pd.DataFrame:
    rv = clean["product_review"]
    return (rv.groupby("customer_id")
              .agg(n_reviews=("id", "count"), avg_star_rating=("star_rating", "mean"),
                   low_rating_share=("star_rating", lambda s: (s <= 2).mean()))
              .reset_index())


def build_crm_features(clean: dict) -> pd.DataFrame:
    crm = clean["crm"]
    counts = pd.crosstab(crm["customer_id"], crm["type"]).rename(columns={"complaint": "n_complaints", "query": "n_queries"})
    counts["main_contact_platform"] = crm.groupby("customer_id")["platform"].agg(lambda s: s.mode().iat[0])
    return counts.reset_index()


def build_engagement_features(clean: dict, snapshot: pd.Timestamp) -> pd.DataFrame:
    lg = clean["server_log"]
    f = lg.groupby("login").agg(n_logins=("session_id", "nunique"), n_devices=("device_model", "nunique"),
                                n_ips=("ip", "nunique"), last_login=("datetime", "max"),
                                main_device_type=("device_type", lambda s: s.mode().iat[0]))
    f["days_since_last_login"] = (snapshot - f.pop("last_login")).dt.days
    return f.reset_index().rename(columns={"login": "email"})


def build_survey_features(clean: dict) -> pd.DataFrame:
    sv = clean["survey"].copy()
    sv["income_band"] = pd.cut(sv["income_gbp"], bins=[0, 30_000, 40_000, 50_000, 75_000, np.inf],
                               labels=["<£30k", "£30-40k", "£40-50k", "£50-75k", "£75k+"], right=False)
    sv["nps_category"] = pd.cut(sv["nps_recommend"], bins=[-1, 6, 8, 10], labels=["Detractor", "Passive", "Promoter"])
    return sv[["email", "reported_income", "income_gbp", "income_band", "nps_recommend", "nps_category",
               "satisfaction_score", "online_comparison_freq"]]

In [ ]:
def build_customer_dataset(clean: dict) -> pd.DataFrame:
    """Join every feature group onto the subscription base (left joins keep all 221 customers)."""
    snapshot = get_snapshot_date(clean)
    log.info("Snapshot date: %s", snapshot)

    df = (build_subscription_features(clean, snapshot)
          .merge(build_sales_features(clean, snapshot), on="customer_id", how="left", validate="one_to_one")
          .merge(build_review_features(clean), on="customer_id", how="left", validate="one_to_one")
          .merge(build_crm_features(clean), on="customer_id", how="left", validate="one_to_one")
          .merge(build_engagement_features(clean, snapshot), on="email", how="left", validate="one_to_one")
          .merge(build_survey_features(clean), on="email", how="left", validate="one_to_one"))

    # ---- Missing-value handling: "no activity" is a real zero, not unknown ----
    zero_fill = ["n_orders", "total_spend_gbp", "n_items", "n_categories", "n_reviews",
                 "n_complaints", "n_queries", "n_logins", "n_devices", "n_ips"]
    df[zero_fill] = df[zero_fill].fillna(0).astype(int)
    # Customers with no shop orders (only the subscription charge): no spend, no late deliveries,
    # and "days since last order" = their whole tenure
    df["has_orders"] = (df["n_orders"] > 0).astype(int)
    df[["avg_order_value_gbp", "late_delivery_rate", "avg_delivery_delay_days"]] = \
        df[["avg_order_value_gbp", "late_delivery_rate", "avg_delivery_delay_days"]].fillna(0)
    df["days_since_last_order"] = df["days_since_last_order"].fillna(df["tenure_days"])
    for col in ["favourite_payment", "favourite_category"]:
        df[col] = df[col].fillna("none")
    df["has_reviews"] = (df["n_reviews"] > 0).astype(int)
    df["had_crm_contact"] = ((df["n_complaints"] + df["n_queries"]) > 0).astype(int)
    df["main_contact_platform"] = df["main_contact_platform"].fillna("none")
    # Rating is genuinely unknown if no reviews -> impute median but keep the has_reviews flag
    df["avg_star_rating"] = df["avg_star_rating"].fillna(df["avg_star_rating"].median())
    df["low_rating_share"] = df["low_rating_share"].fillna(df["low_rating_share"].median())

    # ---- Rate features (normalise by tenure) ----
    m = df["tenure_months"]
    df["orders_per_month"] = df["n_orders"] / m
    df["spend_per_month_gbp"] = df["total_spend_gbp"] / m
    df["logins_per_month"] = df["n_logins"] / m
    df["complaints_per_month"] = df["n_complaints"] / m
    df["queries_per_month"] = df["n_queries"] / m
    contacts = df["n_complaints"] + df["n_queries"]
    df["complaint_share"] = np.where(contacts > 0, df["n_complaints"] / contacts.replace(0, np.nan), 0.0)

    # ---- Types: categoricals save space in Parquet and keep the order of categories ----
    for col in ["county", "subscription_type", "delivery_service", "favourite_payment", "favourite_category",
                "main_contact_platform", "main_device_type"]:
        df[col] = df[col].astype("category")
    df["subscription_type"] = df["subscription_type"].cat.reorder_categories(["SuperSaver", "Basic", "Premium"], ordered=True)

    df = df.drop(columns=["email"])            # PII: customer_id is the key from here on
    return df.sort_values("customer_id").reset_index(drop=True)


churn_df = build_customer_dataset(clean)
print(churn_df.shape)
churn_df.head()

In [ ]:
def encode_for_modelling(df: pd.DataFrame) -> pd.DataFrame:
    """Model-ready copy: one-hot encode categoricals, standardise (z-score) numeric features, drop identifiers/dates/leakage."""
    drop = ["customer_id", "start_date", "end_date", "start_month", "end_month", "auto_renew",   # auto_renew IS the target
            "reported_income", "county",                                                         # county: 40+ levels, too sparse
            # LEAKAGE: churned customers stop ordering/logging in AFTER they cancel, so recency describes
            # the outcome rather than predicting it (corr ~0.75 with churn). Kept in the analysis file, excluded here.
            "days_since_last_order", "days_since_last_login",
            "n_ips", "tenure_months"]                                                            # duplicates n_logins / tenure_days
    X = df.drop(columns=drop)
    X = X.loc[:, X.nunique() > 1]                                                                # drop zero-variance columns (e.g. n_devices)
    y = X.pop("churned")
    cat_cols = X.select_dtypes(include=["category", "object"]).columns
    num_cols = X.columns.difference(cat_cols)
    X_num = X[num_cols].astype(float)
    X_num = (X_num - X_num.mean()) / X_num.std(ddof=0).replace(0, 1)
    X_cat = pd.get_dummies(X[cat_cols].astype(str), prefix=cat_cols, dtype=int)
    out = pd.concat([df[["customer_id"]], X_num, X_cat], axis=1)
    out["churned"] = y.values
    return out


model_df = encode_for_modelling(churn_df)
print(model_df.shape)

## 4. Load  *(owner: Pravalika)*

**Why Parquet?** It is a columnar, compressed (snappy) and *typed* format. Datetimes, categoricals and numeric types survive the round trip, unlike CSV,
and it is the standard format for analytics and data lakes. After writing, `load_parquet` **reads the file back and compares it with the
in-memory frame**, so we only report success if the file on disk is correct.

In [ ]:
def load_parquet(df: pd.DataFrame, path: Path) -> Path:
    """Write df to Parquet, then read it back and verify shape, columns and contents."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, engine="pyarrow", compression="snappy", index=False)
    except Exception as exc:
        log.error("Failed to write %s: %s", path, exc)
        raise
    check = pd.read_parquet(path)
    assert check.shape == df.shape, f"Shape mismatch after write: {check.shape} vs {df.shape}"
    assert list(check.columns) == list(df.columns), "Column mismatch after write"
    pd.testing.assert_frame_equal(check, df, check_dtype=False, check_categorical=False)
    log.info("Loaded %s  (%d rows x %d cols, %.1f KB)", path, *df.shape, path.stat().st_size / 1024)
    return path


load_parquet(churn_df, OUTPUT_FILE)
load_parquet(model_df, MODEL_READY_FILE)

In [ ]:
def run_pipeline(sources: dict = SOURCES) -> pd.DataFrame:
    """Full ETL in one call: Extract -> Quality checks -> Clean -> Feature engineering -> Load."""
    raw = extract_all(sources)
    report = run_quality_checks(raw)
    if (report["status"] == "ERROR").any():
        raise ValueError("Pipeline stopped: data quality errors")
    clean = clean_all(raw)
    df = build_customer_dataset(clean)
    load_parquet(df, OUTPUT_FILE)
    load_parquet(encode_for_modelling(df), MODEL_READY_FILE)
    return df

# Data dictionary for the final dataset
data_dictionary = pd.DataFrame({
    "dtype": churn_df.dtypes.astype(str),
    "non_null": churn_df.notna().sum(),
    "example": churn_df.iloc[0],
})
data_dictionary